# Data Treatment and Cleaning for Comments

This notebook performs cleaning and validation of the YouTube comments dataset, preparing it for downstream analyses.

**Pipeline**: Language filtering → Date validation → Missing value handling → Remove non-informative columns → Deduplication

**Input**: `final_comments_match.csv` (raw comments)
**Output**: `final_comments_match_cleaned.csv` (cleaned and validated comments)

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from tqdm import tqdm
from transformers import pipeline
import time
from wordcloud import WordCloud
from nltk.corpus import stopwords
import nltk
import re
from scipy import stats
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from scipy.stats import zscore
import warnings
import os

# Project paths
PROJECT_ROOT = os.path.abspath(os.getcwd())
CLEANED_DATA_DIR = os.path.join(PROJECT_ROOT, 'cleaned_data')
FIGS_DIR = os.path.join(PROJECT_ROOT, 'figs')
os.makedirs(FIGS_DIR, exist_ok=True)

STOPWORDS_PT = set(stopwords.words('portuguese'))

---
## 1. Loading and Language Filtering

Load comments dataset and keep only videos classified as Portuguese:
- **Source**: List of Portuguese video IDs (`ids_portuguese.txt`) produced by automatic language detection
- **Filtering**: Remove comments from videos in other languages to focus analysis on Portuguese content

In [ ]:
df = pd.read_csv(os.path.join(CLEANED_DATA_DIR, 'final_commments_match.csv'))
df.head()

In [ ]:
df.iloc[:5, [9, 11, 12]]

In [ ]:
df.shape

In [ ]:
df['video_id'].nunique()

In [ ]:
with open(os.path.join(CLEANED_DATA_DIR, 'ids_portuguese.txt'), 'r') as f:
    ids = f.read().splitlines()

In [ ]:
df = df[df['video_id'].isin(ids)]

In [ ]:
df.shape

In [ ]:
df['video_id'].nunique()

In [ ]:
df.info()

In [ ]:
df.loc[(df['language_langdetect'].notna()), ['language', 'language_langdetect']].head()

In [ ]:
df.loc[(df['language'] != 'pt') & (df['language_langdetect'] != 'pt')].shape

In [ ]:
df[(df['language_langdetect'].isna())].shape

In [ ]:
df[df['language'] != 'pt'].shape

In [ ]:
df[df['language_langdetect'] == df['language']].shape

In [ ]:
df.isna().sum()

In [ ]:
df.head()

In [ ]:
df['published_at'] = pd.to_datetime(df['published_at'], utc=True, errors='coerce')
df['updated_at'] = pd.to_datetime(df['updated_at'], utc=True, errors='coerce')

---
## 2. Date Validation and Handling

Convert date columns to `datetime` and remove invalid records:
- **published_at / updated_at**: convert to datetime with UTC timezone
- **Filter**: Remove comments dated in 2025 (incomplete year)
- **Missing dates**: identify and remove records with missing or invalid dates

In [ ]:
df[df['published_at'].isna() | df['updated_at'].isna()].shape

In [ ]:
df.isna().sum()

In [ ]:
df = df[(df['published_at'].dt.year != 2025) & (df['updated_at'].dt.year != 2025)]

In [ ]:
df[df['published_at'].dt.year == 2025].shape

In [ ]:
# Since 'language_langdetect' only has values where 'language' is different from 'pt', I will fill missing values with 'rob_pt' for Portuguese videos
df.loc[df['language_langdetect'].isna() & (df['language'] == 'pt'), 'language_langdetect'] = 'rob_pt'

---
## 3. Missing Value Handling

Handle missing data consistently:
- **language_langdetect**: fill with 'rob_pt' when `language == 'pt'` (indicates RoBERTa detection)
- **parent_id**: fill with 'root' for top-level comments (non-replies)
- **Removal**: drop rows with critical missing values (dates, IDs)
- **Validation**: ensure consistency between `parent_id` and `is_reply`

In [ ]:
df.loc[df['parent_id'].isna(), ['parent_id', 'comment']].head()

In [ ]:
df.loc[(df['parent_id'].isna()) & (df['is_reply'] == False)].shape

In [ ]:
# All comments without parent_id are top-level (root) comments
print('Total comments without parent_id:', df.loc[df['parent_id'].isna(), 'comment'].count())
print('Total comments without parent_id and not replies:', df.loc[(df['parent_id'].isna()) & (df['is_reply'] == False), 'comment'].count())
print('Total comments with missing is_reply info:', df['is_reply'].isna().sum())
print('Difference: ', df.loc[df['parent_id'].isna(), 'comment'].count() - df.loc[(df['parent_id'].isna()) & (df['is_reply'] == False), 'comment'].count())

In [ ]:
# Because many parent_id values are missing, assume these comments are roots and fill with 'root'
df.loc[(df['parent_id'].isna()) & (df['is_reply'] == False), 'parent_id'] = 'root'

In [ ]:
df.isna().sum()

In [ ]:
df.loc[df['published_at'].isna()].head()

In [ ]:
# Most missing data seem to be for this specific comment
outros_sem_data = df.loc[(df['published_at'].isna()) & (df['comment_id'] != 'UgzNiQdoxnPl3oORAed4AaABAg')]
print(f"Quantity: {len(outros_sem_data)}")

In [ ]:
# In addition to the previous case, these 3 comments also have missing data, but even for the filled info they seem wrong.
outros_sem_data.head()

In [ ]:
df.dropna(inplace=True)

In [ ]:
df.isna().sum()

In [ ]:
df['viewer_rating'].value_counts()

In [ ]:
# Final df shape (525370, 17), in total 26,850 comments were removed due to missing data
df.shape

In [ ]:
# Minimum and maximum date for published_at
print('Minimum date for published_at:', df['published_at'].min())
print('Maximum date for published_at:', df['published_at'].max())

# Minimum and maximum date for updated_at
print('\nMinimum date for updated_at:', df['updated_at'].min())
print('Maximum date for updated_at:', df['updated_at'].max())

In [ ]:
# Only value for viewer_rating is 'none', column doesn't seem to add much.
print(df['viewer_rating'].value_counts())
df.drop(columns=['viewer_rating'], inplace=True)

---
## 4. Final Cleaning and Optimization

Remove non-informative columns and deduplicate:
- **Removed columns**: `viewer_rating` (always 'none'), `can_rate` (always True)
- **Type conversions**: convert `like_count` and `is_reply` to `int`
- **Deduplication**: drop duplicate comments based on `comment_id`
- **Final validation**: check counts of unique videos, channels, comments and users
- **Output**: cleaned dataset saved as `final_comments_match_cleaned.csv` (~525k comments)

In [ ]:
# Only value for can_rate is True, column doesn't seem to add much.
print(df['can_rate'].value_counts())
df.drop(columns=['can_rate'], inplace=True)

In [ ]:
df['like_count'] = df['like_count'].astype(int)
df['is_reply'] = df['is_reply'].astype(int)

In [ ]:
df.shape

In [ ]:
df.drop_duplicates(subset='comment_id', inplace=True)

In [ ]:
df.duplicated().sum()

In [ ]:
df.head()

In [ ]:
print('Number of unique videos:', df['video_id'].nunique())
print('Number of unique channels:', df['channel_id'].nunique())
print('Number of unique comments:', df['comment_id'].nunique())
print('Number of unique users:', df['author_channel_id'].nunique())

In [ ]:
df.to_csv(os.path.join(CLEANED_DATA_DIR, 'final_comments_match_cleaned.csv'), index=False)